# 03 — Side-by-Side Comparison Video Generator
**Paper:** *Markov Logic Process: Augmenting Reinforcement Learning with Symbolic
Association-Rule Reasoning via the Logos Module*
**Authors:** Saiyam Jain · Swaroop Bhowmik · Dipanjan Choudhury · Santosh Kumar Sahoo

### What this notebook produces
One MP4 per environment (LL-Std, LL-Wind), showing **MDP vs MLP-Full** side-by-side.
For each pair of frames:
- **Left panel (MDP):** raw rendered frame + Q-values bar chart
- **Right panel (MLP-Full):** rendered frame + Q-values + Φ_L bars + top-3 active rules overlay
- **Top banner:** episode stats, seed, cumulative reward, rule count, U_L

Workflow:
1. Load best-seed models from `checkpoints/models/`
2. Re-instantiate agents, load weights
3. Rollout both agents simultaneously from the same seed
4. Render each frame pair using Pillow / matplotlib
5. Assemble into MP4 via imageio-ffmpeg

### Comparison seeds
`COMPARISON_SEEDS = [42, 123, 456]` — produces 3 videos per environment (6 total).

## 0 · Install & Imports

In [ ]:
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

pip('gymnasium[box2d]>=0.29.0')
pip('mlxtend>=0.23.0')
pip('imageio[ffmpeg]')
pip('opencv-python-headless')

import os, sys, json, random, warnings, time, textwrap
from pathlib import Path
from collections import deque
from concurrent.futures import ThreadPoolExecutor
import threading

import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')   # headless
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import imageio
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from PIL import Image, ImageDraw, ImageFont
import warnings; warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE   = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
PATHS  = {
    'models': BASE/'checkpoints'/'models',
    'videos': BASE/'checkpoints'/'videos',
    'figs':   BASE/'checkpoints'/'figures',
}
for p in PATHS.values(): p.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}  |  Videos will be saved to: {PATHS['videos']}")

## 1 · Re-define Agent Classes (self-contained)

In [ ]:
# ── Shared network & buffer ───────────────────────────────────────────────────
class QNet(nn.Module):
    def __init__(self, obs_dim=8, n_act=4, hidden=(64,64)):
        super().__init__()
        layers,d=[],obs_dim
        for h in hidden: layers+=[nn.Linear(d,h),nn.ReLU()]; d=h
        layers.append(nn.Linear(d,n_act)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

class ReplayBuffer:
    def __init__(self,cap=10_000): self._b=deque(maxlen=cap)
    def push(self,o,a,r,no,d): self._b.append((o,a,r,no,d))
    def sample(self,bs):
        b=random.sample(self._b,bs); o,a,r,no,d=zip(*b)
        return (np.array(o,np.float32),np.array(a,np.int64),np.array(r,np.float32),
                np.array(no,np.float32),np.array(d,np.float32))
    def __len__(self): return len(self._b)

# ── Discretiser ───────────────────────────────────────────────────────────────
FEAT_NAMES=['x','y','x_vel','y_vel','angle','ang_vel']
BIN_LABELS=['Low','Med','High']
CONT_IDX=[0,1,2,3,4,5]

class Discretiser:
    def __init__(self):
        self._buf=[]
        self._min=np.array([-1.5,-0.5,-2.5,-2.5,-3.14,-5.0])
        self._max=np.array([ 1.5, 1.5, 2.5, 2.5, 3.14, 5.0])
        self._q33=self._min+(self._max-self._min)/3
        self._q66=self._min+2*(self._max-self._min)/3

    def partial_fit(self,obs):
        self._buf.append(np.asarray(obs)[CONT_IDX])
        if len(self._buf)>=200:
            arr=np.stack(self._buf)
            self._min=np.minimum(self._min,arr.min(0))
            self._max=np.maximum(self._max,arr.max(0))
            self._q33=np.percentile(arr,33,axis=0)
            self._q66=np.percentile(arr,66,axis=0)
            self._buf.clear()

    def transform(self,obs):
        c=np.asarray(obs)[CONT_IDX]
        ew=np.clip(((c-self._min)/(self._max-self._min+1e-8)*3).astype(int),0,2)
        ef=np.zeros(6,int); ef[c>=self._q33]=1; ef[c>=self._q66]=2
        b=np.clip(np.round((ew+ef)/2).astype(int),0,2)
        return [(FEAT_NAMES[i],BIN_LABELS[b[i]]) for i in range(6)]

    def bin_reward(self,r): return 'Low' if r<-50 else ('High' if r>50 else 'Med')

    def itemset(self,obs,action,reward=None):
        items=[f'{n}:{l}' for n,l in self.transform(obs)]
        items.append(f'action:{action}')
        if reward is not None: items.append(f'reward:{self.bin_reward(reward)}')
        return items

# ── Logos module ──────────────────────────────────────────────────────────────
class Logos:
    def __init__(self,lam=1.5,win=2000,delta=2000,sig=0.20,kap=0.80,pcap=5,dth=0.90,n_act=4):
        self.lam=lam; self.win=win; self.delta=delta; self.sig=sig; self.kap=kap
        self.pcap=pcap; self.dth=dth; self.n_act=n_act
        self.disc=Discretiser()
        self._window=deque(maxlen=win); self._step=0
        self._rules=[]; self._persist={}; self._UL=0.0
        self._lock=threading.Lock()
        self._exec=ThreadPoolExecutor(max_workers=1); self._busy=False

    def add_exp(self,obs,action,reward):
        self.disc.partial_fit(obs)
        self._window.append(self.disc.itemset(obs,action,reward))
        self._step+=1
        if self._step%self.delta==0 and len(self._window)>=50 and not self._busy:
            self._busy=True; self._exec.submit(self._mine)

    def _mine(self):
        try:
            snap=list(self._window); valid=self._apriori(snap); self._upd(valid)
        finally: self._busy=False

    def _apriori(self,trans):
        if len(trans)<20: return []
        te=TransactionEncoder()
        try: arr=te.fit_transform(trans)
        except: return []
        df=pd.DataFrame(arr,columns=te.columns_)
        try: freq=apriori(df,min_support=self.sig,use_colnames=True,verbose=0)
        except: return []
        if freq.empty: return []
        try: rdf=association_rules(freq,metric='confidence',min_threshold=self.kap)
        except: return []
        if rdf.empty: return []
        out=[]
        for _,row in rdf.iterrows():
            ant,con=row['antecedents'],row['consequents']
            ant_nr={i for i in ant if not i.startswith('reward:')}
            if not ant_nr: continue
            if not any(i.startswith('action:') or i.startswith('reward:') for i in con): continue
            out.append({'ant':ant,'con':con,'conf':float(row['confidence']),'pers':0})
        return out

    def _rkey(self,r): return ','.join(sorted(r['ant']))+'->'+','.join(sorted(r['con']))

    def _upd(self,new_rules):
        nk={self._rkey(r):r for r in new_rules}
        upd={}
        for k,r in nk.items(): upd[k]=self._persist.get(k,0)+1; r['pers']=upd[k]
        ul=sum(1 for v in upd.values() if v>=3)/len(nk) if nk else 0.0
        with self._lock: self._rules=list(nk.values()); self._persist=upd; self._UL=ul

    def phi(self,obs,action):
        with self._lock: rules=list(self._rules); UL=self._UL
        if not rules: return 0.0
        items=set(self.disc.itemset(obs,action))
        s=sum(r['conf']*min(r['pers'],self.pcap) for r in rules
              if frozenset(i for i in r['ant'] if not i.startswith('reward:')).issubset(items))
        return self.lam*UL*s

    def phi_all(self,obs): return np.array([self.phi(obs,a) for a in range(self.n_act)])

    def explain(self,obs,action):
        with self._lock: rules=list(self._rules); UL=self._UL
        if not rules: return 0.0,[]
        items=set(self.disc.itemset(obs,action))
        matched=[]; total=0.0
        for r in rules:
            ant_s=frozenset(i for i in r['ant'] if not i.startswith('reward:'))
            if ant_s.issubset(items):
                c=r['conf']*min(r['pers'],self.pcap); total+=c
                matched.append({'ant':' ∧ '.join(sorted(ant_s)),
                                 'con':' ∨ '.join(sorted(r['con'])),
                                 'conf':r['conf'],'pers':r['pers'],
                                 'contrib':self.lam*UL*c})
        matched.sort(key=lambda x:x['contrib'],reverse=True)
        return self.lam*UL*total, matched[:3]

    def deduction_val(self,obs,q): return self.phi_all(obs) if self._UL>self.dth else q
    @property
    def UL(self): return self._UL
    @property
    def n_rules(self): return len(self._rules)

# ── Agent wrappers (inference-only) ──────────────────────────────────────────
class MDPInferAgent:
    def __init__(self,device=DEVICE):
        self.q=QNet().to(device); self.dev=device; self.logos=None
    def load(self,path):
        sd=torch.load(path,map_location=self.dev,weights_only=True)
        self.q.load_state_dict(sd['q_net']); self.q.eval()
    def act(self,obs):
        t=torch.tensor(obs,dtype=torch.float32,device=self.dev).unsqueeze(0)
        with torch.no_grad(): q=self.q(t).squeeze(0).cpu().numpy()
        return int(q.argmax()), q, np.zeros(4), []

class MLPFullInferAgent:
    def __init__(self,device=DEVICE):
        self.q=QNet().to(device); self.dev=device; self.logos=Logos()
    def load(self,path):
        sd=torch.load(path,map_location=self.dev,weights_only=True)
        self.q.load_state_dict(sd['q_net']); self.q.eval()
    def act(self,obs):
        t=torch.tensor(obs,dtype=torch.float32,device=self.dev).unsqueeze(0)
        with torch.no_grad(): q=self.q(t).squeeze(0).cpu().numpy()
        q=self.logos.deduction_val(obs,q)
        phi=self.logos.phi_all(obs)
        action=int((q+phi).argmax())
        phi_val,rules=self.logos.explain(obs,action)
        return action, q, phi, rules
    def observe(self,obs,action,reward): self.logos.add_exp(obs,action,reward)

print("Agent classes defined ✅")

## 2 · Frame Rendering Engine

In [ ]:
ACTION_NAMES = ['Do Nothing','Left Thruster','Main Engine','Right Thruster']
QUAL_COLORS  = {'MDP':'#6c757d','MLP-Full':'#4CAF50'}

FW, FH = 460, 400   # rendered game frame size
PANEL_W = 620       # total panel width per agent (frame + stats)
BANNER_H = 64       # top banner height
PADDING  = 8
FONT_SM  = 9; FONT_MD = 11; FONT_LG = 13

def make_q_bar_image(q_vals, phi_vals, chosen_action, agent_type, width=160, height=180):
    """Render a small Q-value (+ Phi-L) bar chart as PIL Image."""
    fig, ax = plt.subplots(figsize=(width/100, height/100), dpi=100)
    x = np.arange(4); w=0.38
    bars_q = ax.bar(x-w/2, q_vals, w, color='#5c85d6', label='Q(s,a)', alpha=0.85)
    if phi_vals is not None and np.any(phi_vals):
        ax.bar(x+w/2, phi_vals, w, color='#ff8c42', label='Φ_L(s,a)', alpha=0.85)
    ax.bar(chosen_action, 0, bottom=0, color='none', edgecolor='red', linewidth=2, width=0.85, zorder=5)
    ax.set_xticks(x); ax.set_xticklabels([f'A{i}' for i in range(4)], fontsize=7)
    ax.tick_params(labelsize=7); ax.legend(fontsize=6, loc='upper right')
    ax.set_title('Q' + (' + Φ_L' if phi_vals is not None and np.any(phi_vals) else ''),
                 fontsize=8, fontweight='bold')
    ax.axhline(0, color='black', lw=0.5)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.tight_layout(pad=0.3)
    fig.canvas.draw()
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(
        fig.canvas.get_width_height()[::-1] + (3,))
    plt.close(fig)
    return Image.fromarray(img).resize((width, height))

def make_rules_overlay(rules, width=610, height=100):
    """Render top-3 matched rules as PIL Image for MLP-Full overlay."""
    img = Image.new('RGB', (width, height), color='#1a1a2e')
    draw = ImageDraw.Draw(img)
    try: font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf', 9)
    except: font = ImageFont.load_default()
    draw.text((4, 3), '🧠 Active Rules (Logos inference):', fill='#ffd700', font=font)
    if not rules:
        draw.text((4, 18), '  (no persistent rules matched yet)', fill='#aaaaaa', font=font)
    else:
        for i, r in enumerate(rules[:3]):
            y = 18 + i*26
            contrib_str = f"+{r['contrib']:.3f}"
            ant_short = r['ant'][:55] + '…' if len(r['ant'])>55 else r['ant']
            draw.text((4, y),   f"R{i+1}: {ant_short}", fill='#90e0ef', font=font)
            draw.text((4, y+11),f"  ⇒ {r['con']}  conf={r['conf']:.2f}  pers={r['pers']}  Δ={contrib_str}",
                      fill='#caf0f8', font=font)
    return img

def render_one_frame(env_frame_rgb, agent_type, q_vals, phi_vals, chosen_action,
                     rules, ep_reward, n_rules, UL, step, seed):
    """
    Compose one panel (460×400 game + stats sidebar) for a single agent.
    Returns PIL Image of size (PANEL_W × (FH + rules_h)).
    """
    rules_h = 100 if agent_type == 'MLP-Full' else 0
    total_h = FH + rules_h + PADDING*2

    panel = Image.new('RGB', (PANEL_W, total_h), '#12121f')
    game_img = Image.fromarray(env_frame_rgb).resize((FW, FH))
    panel.paste(game_img, (PADDING, PADDING))

    # Q/Phi bar chart on the right
    bar_img = make_q_bar_image(q_vals, phi_vals if agent_type=='MLP-Full' else None,
                                chosen_action, agent_type, width=PANEL_W-FW-PADDING*3, height=160)
    panel.paste(bar_img, (FW+PADDING*2, PADDING+10))

    # Stats text on sidebar
    draw = ImageDraw.Draw(panel)
    try:
        f_sm = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', FONT_SM)
        f_md = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', FONT_MD)
    except:
        f_sm = f_md = ImageFont.load_default()

    x_txt = FW + PADDING*2 + 4
    y_s = PADDING + 175
    color_dim = '#aaaaaa'; color_val = '#e0e0e0'; color_hi = '#ffd700'

    col = QUAL_COLORS.get(agent_type, '#ffffff')
    draw.text((x_txt, y_s),    f"{agent_type}", fill=col, font=f_md); y_s+=18
    draw.text((x_txt, y_s),    f"Step:    {step}", fill=color_val, font=f_sm); y_s+=14
    draw.text((x_txt, y_s),    f"Reward:  {ep_reward:+.1f}", fill=color_hi, font=f_sm); y_s+=14
    draw.text((x_txt, y_s),    f"Action:  {ACTION_NAMES[chosen_action]}", fill=color_val, font=f_sm); y_s+=14
    if agent_type == 'MLP-Full':
        draw.text((x_txt, y_s),f"Rules:   {n_rules}", fill=color_val, font=f_sm); y_s+=14
        draw.text((x_txt, y_s),f"U_L:     {UL:.3f}", fill=color_val, font=f_sm); y_s+=14
        draw.text((x_txt, y_s),f"Seed:    {seed}", fill=color_dim, font=f_sm)
        rules_img = make_rules_overlay(rules, width=PANEL_W-PADDING*2, height=rules_h)
        panel.paste(rules_img, (PADDING, FH+PADDING))
    else:
        draw.text((x_txt, y_s),f"Seed:    {seed}", fill=color_dim, font=f_sm)

    return panel

def make_banner(env_name, seed, ep_mdp, ep_mlp, step, total_h):
    """Top banner image."""
    banner = Image.new('RGB', (PANEL_W*2+PADDING, BANNER_H), '#0d0d1a')
    draw = ImageDraw.Draw(banner)
    try:
        fb = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 13)
        fs = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 10)
    except:
        fb = fs = ImageFont.load_default()

    draw.text((PADDING, 5),    f"MLP Paper — {env_name}", fill='#ffd700', font=fb)
    draw.text((PADDING, 22),   f"Seed: {seed}  |  Step: {step}", fill='#cccccc', font=fs)
    draw.text((PADDING, 36),   f"MDP reward: {ep_mdp:+.1f}", fill=QUAL_COLORS['MDP'], font=fs)
    draw.text((PADDING+200,36),f"MLP-Full reward: {ep_mlp:+.1f}", fill=QUAL_COLORS['MLP-Full'], font=fs)
    left_label  = f"← MDP (baseline)"
    right_label = f"MLP-Full (proposed) →"
    mid = PANEL_W*2 // 2
    draw.text((mid-130, 5), left_label,  fill=QUAL_COLORS['MDP'],      font=fb)
    draw.text((mid+30,  5), right_label, fill=QUAL_COLORS['MLP-Full'], font=fb)
    return banner

def assemble_frame(env_name, seed, left_panel, right_panel, ep_mdp, ep_mlp, step):
    """Combine banner + two panels into one video frame."""
    pw, ph = left_panel.size
    banner = make_banner(env_name, seed, ep_mdp, ep_mlp, step, ph)
    frame  = Image.new('RGB', (pw*2+PADDING, BANNER_H+ph), '#0d0d1a')
    frame.paste(banner, (0, 0))
    frame.paste(left_panel,  (0,            BANNER_H))
    frame.paste(right_panel, (pw+PADDING,   BANNER_H))
    return np.array(frame)

print("Rendering engine defined ✅")

## 3 · Generate Comparison Videos

In [ ]:
# Configuration
EXPERIMENT_ENVS = {
    'll_std':  {'display_name':'LunarLander (standard)',          'kwargs':{}},
    'll_wind': {'display_name':'LunarLander (wind + turbulence)',
                'kwargs':{'enable_wind':True,'wind_power':15.0,'turbulence_power':1.5}},
}
COMPARISON_SEEDS = [42, 123, 456]   # 3 seeds = 3 videos per env (6 total)
VIDEO_FPS    = 30
MAX_STEPS    = 1200   # max steps per episode video (longer = bigger file)

def load_agent_for_video(agent_class, model_path):
    """Load a trained agent from disk.  Returns None if file missing."""
    if not Path(model_path).exists():
        print(f"  ⚠️  Model not found: {model_path} — skipping")
        return None
    agent = agent_class()
    agent.load(model_path)
    return agent

def rollout_and_render(env_cfg, seed, mdp_agent, mlp_agent):
    """
    Run both agents step-by-step from same seed, collect rendered frames.
    Returns list of numpy arrays (video frames).
    """
    env_mdp  = gym.make('LunarLander-v3', render_mode='rgb_array', **env_cfg.get('kwargs',{}))
    env_mlp  = gym.make('LunarLander-v3', render_mode='rgb_array', **env_cfg.get('kwargs',{}))

    obs_mdp, _ = env_mdp.reset(seed=seed)
    obs_mlp, _ = env_mlp.reset(seed=seed)

    frames = []; ep_r_mdp = 0.0; ep_r_mlp = 0.0
    done_mdp = done_mlp = False
    step = 0

    while (not done_mdp or not done_mlp) and step < MAX_STEPS:
        # ── MDP step ─────────────────────────────────────────────────────
        if not done_mdp:
            a_mdp, q_mdp, phi_mdp, rules_mdp = mdp_agent.act(obs_mdp)
            no_mdp, r_mdp, t_mdp, tr_mdp, _ = env_mdp.step(a_mdp)
            ep_r_mdp += r_mdp
            frame_mdp = env_mdp.render()
            if t_mdp or tr_mdp: done_mdp = True
            obs_mdp = no_mdp
        else:
            frame_mdp = np.zeros((400,600,3),np.uint8); a_mdp=0
            q_mdp=np.zeros(4); phi_mdp=np.zeros(4); rules_mdp=[]

        # ── MLP-Full step ─────────────────────────────────────────────────
        if not done_mlp:
            a_mlp, q_mlp, phi_mlp, rules_mlp = mlp_agent.act(obs_mlp)
            mlp_agent.observe(obs_mlp, a_mlp, r_mdp)   # warm Logos window
            no_mlp, r_mlp, t_mlp, tr_mlp, _ = env_mlp.step(a_mlp)
            ep_r_mlp += r_mlp
            frame_mlp = env_mlp.render()
            if t_mlp or tr_mlp: done_mlp = True
            obs_mlp = no_mlp
        else:
            frame_mlp = np.zeros((400,600,3),np.uint8); a_mlp=0
            q_mlp=np.zeros(4); phi_mlp=np.zeros(4); rules_mlp=[]

        # ── Render panels ────────────────────────────────────────────────
        left  = render_one_frame(frame_mdp, 'MDP', q_mdp, None, a_mdp,
                                  [], ep_r_mdp, 0, 0.0, step, seed)
        right = render_one_frame(frame_mlp, 'MLP-Full', q_mlp, phi_mlp, a_mlp,
                                  rules_mlp, ep_r_mlp,
                                  getattr(mlp_agent.logos,'n_rules',0),
                                  getattr(mlp_agent.logos,'UL',0.0), step, seed)
        composed = assemble_frame(env_cfg['display_name'], seed, left, right,
                                   ep_r_mdp, ep_r_mlp, step)
        frames.append(composed)
        step += 1

    env_mdp.close(); env_mlp.close()
    print(f"    Rendered {step} frames | MDP={ep_r_mdp:.1f} | MLP={ep_r_mlp:.1f}")
    return frames

# ── Main video generation loop ────────────────────────────────────────────────
generated = []
for ek, ecfg in EXPERIMENT_ENVS.items():
    for seed in COMPARISON_SEEDS:
        mdp_path  = PATHS['models'] / f"MDP_{ek}_seed{seed}.pt"
        mlp_path  = PATHS['models'] / f"MLP_Full_{ek}_seed{seed}.pt"

        # Fall back to best-seed model if individual seed not found
        if not mdp_path.exists():  mdp_path  = PATHS['models'] / f"MDP_{ek}_best.pt"
        if not mlp_path.exists():  mlp_path  = PATHS['models'] / f"MLP_Full_{ek}_best.pt"

        print(f"\n▶ {ek} / seed {seed}")
        mdp_agent = load_agent_for_video(MDPInferAgent, mdp_path)
        mlp_agent = load_agent_for_video(MLPFullInferAgent, mlp_path)

        if mdp_agent is None or mlp_agent is None:
            print(f"  Skipping — model file missing. Run 01_train.ipynb first.")
            continue

        print(f"  Rolling out ...")
        frames = rollout_and_render(ecfg, seed, mdp_agent, mlp_agent)

        if not frames:
            print("  No frames generated — skipping.")
            continue

        out_path = PATHS['videos'] / f"comparison_{ek}_seed{seed}.mp4"
        with imageio.get_writer(str(out_path), fps=VIDEO_FPS, quality=8) as writer:
            for f in frames: writer.append_data(f)

        size_mb = out_path.stat().st_size / 1e6
        print(f"  ✅ Saved: {out_path}  ({size_mb:.1f} MB, {len(frames)} frames @ {VIDEO_FPS}fps)")
        generated.append(out_path)

print()
print(f"═══ Video generation complete — {len(generated)} videos produced ═══")
for p in generated: print(f"  {p}")

## 4 · Best-Seed Summary Videos

In [ ]:
# Also generate one "highlight" video per env using the best-seed models
print("Generating best-seed highlight videos ...")
for ek, ecfg in EXPERIMENT_ENVS.items():
    mdp_path = PATHS['models'] / f"MDP_{ek}_best.pt"
    mlp_path = PATHS['models'] / f"MLP_Full_{ek}_best.pt"

    mdp_agent = load_agent_for_video(MDPInferAgent, mdp_path)
    mlp_agent = load_agent_for_video(MLPFullInferAgent, mlp_path)

    if mdp_agent is None or mlp_agent is None:
        print(f"  Skipping {ek} — best model missing."); continue

    print(f"  {ek} (best-seed highlight) ...")
    frames = rollout_and_render(ecfg, seed=42, mdp_agent=mdp_agent, mlp_agent=mlp_agent)

    out = PATHS['videos'] / f"highlight_{ek}_best.mp4"
    with imageio.get_writer(str(out), fps=VIDEO_FPS, quality=9) as writer:
        for f in frames: writer.append_data(f)
    print(f"  ✅ Saved: {out}")

print()
print("Done. Video files are in:", PATHS['videos'])
print("Upload to GitHub Releases or Hugging Face Hub for sharing.")